# Spot the Screen Photo


## 1. Install dependencies

In [ ]:
!pip install -q opencv-python-headless PyWavelets scikit-learn joblib scipy scikit-image
print("Installed")

Installed


## 2. Uploading the dataset


In [ ]:
import os
import zipfile

os.makedirs("data", exist_ok=True)

from google.colab import files
print("Uploaded zip fileee")
uploaded = files.upload()

for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as zip_ref:
        zip_ref.extractall("data")

REAL_DIR = "data/real"
SCREEN_DIR = "data/fake"

VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

n_real = len([f for f in os.listdir(REAL_DIR) if os.path.splitext(f)[1].lower() in VALID_EXT]) if os.path.isdir(REAL_DIR) else 0
n_screen = len([f for f in os.listdir(SCREEN_DIR) if os.path.splitext(f)[1].lower() in VALID_EXT]) if os.path.isdir(SCREEN_DIR) else 0
print(f"real/   : {n_real} images")
print(f"screen/ : {n_screen} images")
assert n_real > 0 and n_screen > 0, "Could not find data/real and data/screen - check your zip folder names."


Uploaded zip fileee


Saving fake.zip to fake.zip
Saving real.zip to real.zip
real/   : 50 images
screen/ : 50 images


## 3. Feature extraction code



In [ ]:
import cv2
import numpy as np
import pywt
from scipy.stats import kurtosis, skew, entropy as scipy_entropy
from scipy.ndimage import uniform_filter
from skimage.feature import local_binary_pattern
import warnings
warnings.filterwarnings("ignore")

TEXTURE_LONG_SIDE = 384
STATS_LONG_SIDE = 200


def load_resized(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Cannot read image: {path}")
    return img


def _resize_long_side(img, target):
    h, w = img.shape[:2]
    long_side = max(h, w)
    if long_side > target:
        scale = target / long_side
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return img


def _fast_entropy(hist):

    h = hist[hist > 0]
    return float(-np.sum(h * np.log(h)))


# ── Signal 1: wavelet ───────────────────────────────────────────────
def wavelet_features(gray):
    g = (gray / 255.0).astype(np.float32)
    coeffs = pywt.wavedec2(g, wavelet="db4", level=3)
    feats = []
    energies = []
    for level in coeffs[1:]:
        level_energy = []
        for sb in level:
            f = sb.ravel()
            e = float(np.sum(f.astype(np.float64) ** 2))
            level_energy.append(e)
            hist, _ = np.histogram(f, bins=16, density=True)
            hist = hist + 1e-12
            med = float(np.median(f))
            feats.extend([
                float(np.mean(f)), float(np.var(f)), e,
                _fast_entropy(hist), float(skew(f)), float(kurtosis(f)),
                med, float(np.median(np.abs(f - med))),
                float(np.percentile(np.abs(f), 95)),
            ])
        energies.append(level_energy)
    for i in range(len(energies) - 1):
        for a, b in zip(energies[i], energies[i + 1]):
            feats.append(float(a / (b + 1e-8)))
    return np.array(feats, dtype=np.float32)


# ── Signal 2: moire / FFT peaks ──────────────────────────────────────
def moire_features(gray):
    g = gray / 255.0
    f = np.fft.fftshift(np.fft.fft2(g))
    mag = np.log(np.abs(f) + 1.0)
    H, W = mag.shape
    cy, cx = H // 2, W // 2

    yy, xx = np.ogrid[:H, :W]
    dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    low_mask = dist < (min(H, W) * 0.04)
    search = mag.copy()
    search[low_mask] = 0

    thresh = search.mean() + 3 * search.std()
    peaks = search > thresh
    peak_count = int(peaks.sum())
    peak_strength = float(search[peaks].mean()) if peak_count > 0 else 0.0

    max_r = min(H, W) / 2.0
    nbins = 12
    dist_flat = dist.ravel()
    search_flat = search.ravel()
    radial_idx = np.clip((dist_flat / max_r * nbins).astype(np.int32), 0, nbins - 1)
    radial_sum = np.bincount(radial_idx, weights=search_flat, minlength=nbins)
    radial_cnt = np.bincount(radial_idx, minlength=nbins)
    radial_energy = radial_sum / np.maximum(radial_cnt, 1)
    radial_energy = radial_energy / (radial_energy.sum() + 1e-8)

    theta = np.arctan2(yy - cy, xx - cx)
    nADirs = 8
    theta_flat = theta.ravel()
    low_flat = low_mask.ravel()
    angle_idx = np.clip(((theta_flat + np.pi) / (2 * np.pi) * nADirs).astype(np.int32), 0, nADirs - 1)
    valid = ~low_flat
    dir_sum = np.bincount(angle_idx[valid], weights=search_flat[valid], minlength=nADirs)
    dir_cnt = np.bincount(angle_idx[valid], minlength=nADirs)
    dir_energy = dir_sum / np.maximum(dir_cnt, 1)
    dir_energy = dir_energy / (dir_energy.sum() + 1e-8)
    dir_peak_to_avg = float(dir_energy.max() / (dir_energy.mean() + 1e-8))

    ys, xs = np.where(peaks)
    if len(ys) >= 2:
        coords = np.stack([ys, xs], axis=1).astype(float)
        if len(coords) > 60:
            idx = np.random.RandomState(0).choice(len(coords), 60, replace=False)
            coords = coords[idx]
        diffs = coords[:, None, :] - coords[None, :, :]
        d = np.sqrt((diffs ** 2).sum(-1))
        iu = np.triu_indices(len(coords), k=1)
        spacings = d[iu]
        spacing_std = float(np.std(spacings)) if len(spacings) else 0.0
        spacing_mean = float(np.mean(spacings)) if len(spacings) else 0.0
    else:
        spacing_std, spacing_mean = 0.0, 0.0

    periodic_energy_ratio = float(search[peaks].sum() / (search.sum() + 1e-8)) if peak_count > 0 else 0.0

    feats = [
        peak_count, peak_strength, periodic_energy_ratio,
        dir_peak_to_avg, spacing_mean, spacing_std,
        float(mag.std()), float(mag.max() / (mag.mean() + 1e-8)),
    ]
    feats.extend(radial_energy.tolist())
    feats.extend(dir_energy.tolist())
    return np.array(feats, dtype=np.float32)


# ── Signal 3: LBP ─────────────────────────────────────────────────────
def lbp_features(gray):
    g = gray.astype(np.uint8)
    radius, npoints = 1, 8
    lbp = local_binary_pattern(g, P=npoints, R=radius, method="uniform")
    nbins = npoints + 2
    hist, _ = np.histogram(lbp, bins=nbins, range=(0, nbins), density=True)
    hist = hist + 1e-12
    feats = hist.tolist()
    feats.append(_fast_entropy(hist))
    return np.array(feats, dtype=np.float32)


# ── Signal 4: noise residual ─────────────────────────────────────────
def noise_features(gray):
    g = (gray / 255.0).astype(np.float32)
    k = 5
    local_mean = cv2.boxFilter(g, -1, (k, k), borderType=cv2.BORDER_REFLECT)
    local_sqmean = cv2.boxFilter(g * g, -1, (k, k), borderType=cv2.BORDER_REFLECT)
    local_var = np.maximum(local_sqmean - local_mean * local_mean, 0)
    noise_power = float(local_var.mean())
    gain = local_var / (local_var + noise_power + 1e-8)
    denoised = local_mean + gain * (g - local_mean)
    noise = g - denoised
    flat = noise.ravel()

    H, W = noise.shape
    bs = 16
    Hc, Wc = (H // bs) * bs, (W // bs) * bs
    if Hc >= bs and Wc >= bs:
        blocks = noise[:Hc, :Wc].reshape(Hc // bs, bs, Wc // bs, bs)
        local_vars = blocks.var(axis=(1, 3)).ravel()
    else:
        local_vars = np.array([0.0])

    hp = cv2.Laplacian((g * 255).astype(np.uint8), cv2.CV_64F)
    hp_hist, _ = np.histogram(hp.ravel(), bins=20, density=True)
    hp_hist = hp_hist + 1e-12

    odd_var = float(np.var(noise[::2, :]))
    even_var = float(np.var(noise[1::2, :]))
    col_odd = float(np.var(noise[:, ::2]))
    col_even = float(np.var(noise[:, 1::2]))

    return np.array([
        float(noise.std()), float(kurtosis(flat)), float(skew(flat)),
        float(np.mean(np.abs(flat))), float(np.percentile(np.abs(flat), 90)),
        odd_var / (even_var + 1e-8), col_odd / (col_even + 1e-8),
        float(local_vars.mean()), float(local_vars.std()),
        float(hp.std()), _fast_entropy(hp_hist),
    ], dtype=np.float32)


# ── Signal 5: chromatic aberration ───────────────────────────────────
def chromatic_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    edge_mask = mag > np.percentile(mag, 70)

    r = img[:, :, 2].astype(np.float32)
    g = img[:, :, 1].astype(np.float32)
    b = img[:, :, 0].astype(np.float32)

    feats = []
    for c1, c2 in [(r, g), (g, b), (r, b)]:
        d = np.abs(c1 - c2)[edge_mask]
        if len(d) == 0:
            feats.extend([0.0, 0.0])
        else:
            feats.extend([float(d.mean()), float(d.std())])

    H, W = gray.shape
    cs = int(min(H, W) * 0.2)
    corners = [(slice(0, cs), slice(0, cs)), (slice(0, cs), slice(W-cs, W)),
               (slice(H-cs, H), slice(0, cs)), (slice(H-cs, H), slice(W-cs, W))]
    corner_diffs = []
    for sy, sx in corners:
        d = np.abs(r[sy, sx] - b[sy, sx])
        corner_diffs.append(d.mean())
    feats.append(float(np.mean(corner_diffs)))
    feats.append(float(np.std(corner_diffs)))

    return np.array(feats, dtype=np.float32)


# ── Signal 6: blur / sharpness ───────────────────────────────────────
def sharpness_features(gray):
    g = gray.astype(np.uint8)
    lap = cv2.Laplacian(g, cv2.CV_64F)
    lap_var = float(lap.var())

    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = float(np.mean(gx**2 + gy**2))
    sobel_energy = float(np.sum(np.sqrt(gx**2 + gy**2)))

    edges = cv2.Canny(g, 50, 150)
    edge_density = float(edges.mean() / 255.0)

    mag = np.sqrt(gx**2 + gy**2)
    strong = mag > np.percentile(mag, 90)
    edge_width = float(1.0 / (strong.mean() + 1e-8))

    return np.array([lap_var, tenengrad, sobel_energy, edge_density, edge_width], dtype=np.float32)


# ── Signal 7: reflection / glare (now runs on the small STATS image) ──
def glare_features(img_small):
    hsv = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)

    highlight_mask = v > 240
    highlight_ratio = float(highlight_mask.mean())

    hist, _ = np.histogram(v, bins=32, range=(0, 255), density=True)
    hist = hist + 1e-12
    highlight_entropy = _fast_entropy(hist)

    clip_ratio = float((v >= 254).mean())

    mask_u8 = (highlight_mask.astype(np.uint8)) * 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    glare_areas = stats[1:, cv2.CC_STAT_AREA] if n_labels > 1 else np.array([0])
    glare_mask_area = float(glare_areas.max() / (img_small.shape[0] * img_small.shape[1])) if len(glare_areas) else 0.0

    sat_low = float((s < 30).mean())

    return np.array([highlight_ratio, highlight_entropy, clip_ratio, glare_mask_area, sat_low], dtype=np.float32)


# ── Signal 8: colour-space stats (now runs on the small STATS image) ──
def colorspace_features(img_small):
    feats = []
    hsv = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV).astype(np.float32)
    lab = cv2.cvtColor(img_small, cv2.COLOR_BGR2LAB).astype(np.float32)
    ycc = cv2.cvtColor(img_small, cv2.COLOR_BGR2YCrCb).astype(np.float32)
    bgr = img_small.astype(np.float32)

    for space in [bgr, hsv, lab, ycc]:
        for c in range(3):
            ch = space[:, :, c].ravel()
            feats.extend([float(ch.mean()), float(ch.std()), float(skew(ch))])

    b, g, r = cv2.split(bgr)
    rg = r - g
    yb = 0.5 * (r + g) - b
    colorfulness = float(np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2))

    corr_rg = float(np.corrcoef(r.ravel(), g.ravel())[0, 1])
    corr_gb = float(np.corrcoef(g.ravel(), b.ravel())[0, 1])

    feats.extend([colorfulness, corr_rg, corr_gb])
    return np.array(feats, dtype=np.float32)


def extract_features(img_full):
    """img_full: the raw imread() BGR image, no resizing applied yet."""
    img_tex = _resize_long_side(img_full, TEXTURE_LONG_SIDE)
    img_stats = _resize_long_side(img_full, STATS_LONG_SIDE)
    gray_tex = cv2.cvtColor(img_tex, cv2.COLOR_BGR2GRAY).astype(np.float64)

    f1 = wavelet_features(gray_tex)
    f2 = moire_features(gray_tex)
    f3 = lbp_features(gray_tex)
    f4 = noise_features(gray_tex)
    f5 = chromatic_features(img_tex)
    f6 = sharpness_features(gray_tex)
    f7 = glare_features(img_stats)
    f8 = colorspace_features(img_stats)
    return np.concatenate([f1, f2, f3, f4, f5, f6, f7, f8]).astype(np.float32)

print("Feature extraction methods")


Feature extraction methods


## 4. Load dataset & extract features

In [ ]:
import os, time

DATA_DIR = "data"
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_dataset():
    X, y, paths = [], [], []
    for label, folder in [(0, "real"), (1, "fake")]:
        folder_path = os.path.join(DATA_DIR, folder)
        fnames = sorted(f for f in os.listdir(folder_path)
                         if os.path.splitext(f)[1].lower() in VALID_EXT)
        print(f"Loading {len(fnames)} images from {folder_path} (label={label})")
        for fname in fnames:
            fpath = os.path.join(folder_path, fname)
            try:
                img = load_resized(fpath)
                feats = extract_features(img)
                X.append(feats)
                y.append(label)
                paths.append(fpath)
            except Exception as e:
                print(f"  skipped {fpath}: {e}")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), paths

t0 = time.time()
X, y, paths = load_dataset()
t1 = time.time()
print(f"\nDataset: {X.shape[0]} images, {X.shape[1]} raw features each")
print(f"Class balance: real={int((y==0).sum())}  screen={int((y==1).sum())}")
print(f"Feature extraction took {t1 - t0:.1f}s total -> {(t1 - t0) / max(len(y),1) * 1000:.0f} ms/image average")

Loading 50 images from data/real (label=0)
Loading 50 images from data/fake (label=1)

Dataset: 100 images, 194 raw features each
Class balance: real=50  screen=50
Feature extraction took 12.4s total -> 124 ms/image average


## 5. Train with cross-validation, compare a couple of models




In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

K_BEST = min(18, X.shape[1])

candidates = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(mutual_info_classif, k=K_BEST)),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced", random_state=42)),
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(mutual_info_classif, k=K_BEST)),
        ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("select", SelectKBest(mutual_info_classif, k=K_BEST)),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=5, class_weight="balanced", random_state=42)),
    ]),
}

results = {}
for name, pipe in candidates.items():
    preds = cross_val_predict(pipe, X, y, cv=skf, method="predict")
    probs = cross_val_predict(pipe, X, y, cv=skf, method="predict_proba")[:, 1]
    acc = accuracy_score(y, preds)
    prec = precision_score(y, preds, zero_division=0)
    rec = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    auc = roc_auc_score(y, probs)
    results[name] = dict(acc=acc, prec=prec, rec=rec, f1=f1, auc=auc, preds=preds, probs=probs)
    print(f"{name:20s}  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")

best_name = max(results, key=lambda n: results[n]["acc"])
print(f"\nBest by CV accuracy: {best_name}")

LogisticRegression    acc=0.900  prec=0.900  rec=0.900  f1=0.900  auc=0.955
SVM (RBF)             acc=0.900  prec=0.935  rec=0.860  f1=0.896  auc=0.959
RandomForest          acc=0.900  prec=0.917  rec=0.880  f1=0.898  auc=0.947

Best by CV accuracy: LogisticRegression


In [ ]:

chosen = max(results, key=lambda x: results[x]["acc"])
print("Best model:", chosen)
cm = confusion_matrix(y, results[chosen]["preds"])
print(f"Confusion matrix for {chosen} (rows=true, cols=pred) [0=real, 1=screen]:")
print(cm)
print(f"\nCV accuracy ({N_SPLITS}-fold): {results[chosen]['acc']:.1%}")
print(f"CV ROC-AUC: {results[chosen]['auc']:.3f}")

Best model: LogisticRegression
Confusion matrix for LogisticRegression (rows=true, cols=pred) [0=real, 1=screen]:
[[45  5]
 [ 5 45]]

CV accuracy (5-fold): 90.0%
CV ROC-AUC: 0.955


## 6. Train final model on all data & save




In [ ]:
!pip install joblib

In [ ]:
from joblib import dump
import joblib
import os

os.makedirs("models", exist_ok=True)

final_pipeline = candidates[chosen]

final_pipeline.fit(X, y)

selected_mask = final_pipeline.named_steps["select"].get_support()
print(f"Selected {selected_mask.sum()} / {X.shape[1]} features")
print("Selected feature indices:", np.where(selected_mask)[0].tolist())

joblib.dump(final_pipeline, "models/pipeline_v3.pkl")

print("\n Model saved to models/pipeline_v3.pkl")
print(f"Best model: {chosen}")
print(f"CV Accuracy: {results[chosen]['acc']:.1%}")
print(f"CV ROC-AUC: {results[chosen]['auc']:.3f}")


Selected 18 / 194 features
Selected feature indices: [16, 43, 69, 123, 143, 155, 158, 161, 165, 166, 170, 171, 172, 173, 174, 175, 182, 183]

 Model saved to models/pipeline_v3.pkl
Best model: LogisticRegression
CV Accuracy: 90.0%
CV ROC-AUC: 0.955


## 7. Evaluate — accuracy & latency report




In [ ]:
import time as _time

# Re-extract features end-to-end (load + extract) to get a realistic
# per-image latency that matches what predict.py will see, including
# model inference.
latencies_ms = []
preds_full = []
for p, label in zip(paths, y):
    t0 = _time.time()
    img = load_resized(p)
    feats = extract_features(img)
    score = final_pipeline.predict_proba([feats])[0][1]
    t1 = _time.time()
    latencies_ms.append((t1 - t0) * 1000)
    preds_full.append(1 if score >= 0.5 else 0)

latencies_ms = np.array(latencies_ms)
preds_full = np.array(preds_full)

print(f"Per-image latency: mean={latencies_ms.mean():.1f} ms, "
      f"median={np.median(latencies_ms):.1f} ms, p95={np.percentile(latencies_ms, 95):.1f} ms")
print(f"(measured on {len(paths)} images, this Colab CPU runtime)")

train_set_acc = accuracy_score(y, preds_full)
print(f"\nTrain-set accuracy : {train_set_acc:.1%}")
print(f"Cross-validated accuracy (the honest number to report): {results[chosen]['acc']:.1%}")

Per-image latency: mean=144.5 ms, median=124.9 ms, p95=275.0 ms
(measured on 100 images, this Colab CPU runtime)

Train-set accuracy : 93.0%
Cross-validated accuracy (the honest number to report): 90.0%


In [ ]:
print("Pipeline steps:", [name for name, _ in final_pipeline.steps])
print("Raw feature vector length expected by pipeline:", X.shape[1])
print("Final estimator:", final_pipeline.named_steps["clf"])
print("\npredict.py loads this exact pipeline via:")
print('  joblib.load(os.path.join(_THIS_DIR, "models", "bt_model_v3.pkl"))')
print("and calls model.predict_proba([feats])[0][1] on features from the same")
print("extract_features() pipeline defined above — make sure predict.py's")
print("feature code stays byte-for-byte identical to this notebook's.")

Pipeline steps: ['scaler', 'select', 'clf']
Raw feature vector length expected by pipeline: 194
Final estimator: LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

predict.py loads this exact pipeline via:
  joblib.load(os.path.join(_THIS_DIR, "models", "bt_model_v3.pkl"))
and calls model.predict_proba([feats])[0][1] on features from the same
extract_features() pipeline defined above — make sure predict.py's
feature code stays byte-for-byte identical to this notebook's.


## 9. Download model




In [ ]:
import zipfile

with zipfile.ZipFile("code.zip", "w") as zf:
    zf.write("models/pipeline_v3.pkl")

from google.colab import files
files.download("code.zip")
print(" Downloading")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloading


# **Testing**

In [ ]:
import os
import sys
import warnings
import time
warnings.filterwarnings("ignore")

import cv2
import joblib
import numpy as np
import pywt
from scipy.stats import kurtosis, skew, entropy as scipy_entropy
from skimage.feature import local_binary_pattern
_MODEL_PATH = "/content/models/pipeline_v3.pkl"
IMAGE_PATH = "/content/20260701_012124.heic"

_model = None

# Dual resolution (v4 latency optimization) — see Section 3 above for why.
TEXTURE_LONG_SIDE = 384
STATS_LONG_SIDE = 200


def _load_model():
    global _model
    if _model is None:
        _model = joblib.load(_MODEL_PATH)
    return _model


def _resize_long_side(img, target):
    h, w = img.shape[:2]
    long_side = max(h, w)
    if long_side > target:
        scale = target / long_side
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return img


def _load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Cannot read image: {path}")
    return img


def _fast_entropy(hist):
    h = hist[hist > 0]
    return float(-np.sum(h * np.log(h)))


def _wavelet_features(gray):
    g = (gray / 255.0).astype(np.float32)
    coeffs = pywt.wavedec2(g, wavelet="db4", level=3)
    feats = []
    energies = []
    for level in coeffs[1:]:
        level_energy = []
        for sb in level:
            f = sb.ravel()
            e = float(np.sum(f.astype(np.float64) ** 2))
            level_energy.append(e)
            hist, _ = np.histogram(f, bins=16, density=True)
            hist = hist + 1e-12
            med = float(np.median(f))
            feats.extend([
                float(np.mean(f)), float(np.var(f)), e,
                _fast_entropy(hist), float(skew(f)), float(kurtosis(f)),
                med, float(np.median(np.abs(f - med))),
                float(np.percentile(np.abs(f), 95)),
            ])
        energies.append(level_energy)
    for i in range(len(energies) - 1):
        for a, b in zip(energies[i], energies[i + 1]):
            feats.append(float(a / (b + 1e-8)))
    return np.array(feats, dtype=np.float32)


def _moire_features(gray):
    g = gray / 255.0
    f = np.fft.fftshift(np.fft.fft2(g))
    mag = np.log(np.abs(f) + 1.0)
    H, W = mag.shape
    cy, cx = H // 2, W // 2

    yy, xx = np.ogrid[:H, :W]
    dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    low_mask = dist < (min(H, W) * 0.04)
    search = mag.copy()
    search[low_mask] = 0

    thresh = search.mean() + 3 * search.std()
    peaks = search > thresh
    peak_count = int(peaks.sum())
    peak_strength = float(search[peaks].mean()) if peak_count > 0 else 0.0

    max_r = min(H, W) / 2.0
    nbins = 12
    dist_flat = dist.ravel()
    search_flat = search.ravel()
    radial_idx = np.clip((dist_flat / max_r * nbins).astype(np.int32), 0, nbins - 1)
    radial_sum = np.bincount(radial_idx, weights=search_flat, minlength=nbins)
    radial_cnt = np.bincount(radial_idx, minlength=nbins)
    radial_energy = radial_sum / np.maximum(radial_cnt, 1)
    radial_energy = radial_energy / (radial_energy.sum() + 1e-8)

    theta = np.arctan2(yy - cy, xx - cx)
    nADirs = 8
    theta_flat = theta.ravel()
    low_flat = low_mask.ravel()
    angle_idx = np.clip(((theta_flat + np.pi) / (2 * np.pi) * nADirs).astype(np.int32), 0, nADirs - 1)
    valid = ~low_flat
    dir_sum = np.bincount(angle_idx[valid], weights=search_flat[valid], minlength=nADirs)
    dir_cnt = np.bincount(angle_idx[valid], minlength=nADirs)
    dir_energy = dir_sum / np.maximum(dir_cnt, 1)
    dir_energy = dir_energy / (dir_energy.sum() + 1e-8)
    dir_peak_to_avg = float(dir_energy.max() / (dir_energy.mean() + 1e-8))

    ys, xs = np.where(peaks)
    if len(ys) >= 2:
        coords = np.stack([ys, xs], axis=1).astype(float)
        if len(coords) > 60:
            idx = np.random.RandomState(0).choice(len(coords), 60, replace=False)
            coords = coords[idx]
        diffs = coords[:, None, :] - coords[None, :, :]
        d = np.sqrt((diffs ** 2).sum(-1))
        iu = np.triu_indices(len(coords), k=1)
        spacings = d[iu]
        spacing_std = float(np.std(spacings)) if len(spacings) else 0.0
        spacing_mean = float(np.mean(spacings)) if len(spacings) else 0.0
    else:
        spacing_std, spacing_mean = 0.0, 0.0

    periodic_energy_ratio = float(search[peaks].sum() / (search.sum() + 1e-8)) if peak_count > 0 else 0.0

    feats = [
        peak_count, peak_strength, periodic_energy_ratio,
        dir_peak_to_avg, spacing_mean, spacing_std,
        float(mag.std()), float(mag.max() / (mag.mean() + 1e-8)),
    ]
    feats.extend(radial_energy.tolist())
    feats.extend(dir_energy.tolist())
    return np.array(feats, dtype=np.float32)


def _lbp_features(gray):
    g = gray.astype(np.uint8)
    radius, npoints = 1, 8
    lbp = local_binary_pattern(g, P=npoints, R=radius, method="uniform")
    nbins = npoints + 2
    hist, _ = np.histogram(lbp, bins=nbins, range=(0, nbins), density=True)
    hist = hist + 1e-12
    feats = hist.tolist()
    feats.append(_fast_entropy(hist))
    return np.array(feats, dtype=np.float32)


def _noise_features(gray):
    g = (gray / 255.0).astype(np.float32)
    k = 5
    local_mean = cv2.boxFilter(g, -1, (k, k), borderType=cv2.BORDER_REFLECT)
    local_sqmean = cv2.boxFilter(g * g, -1, (k, k), borderType=cv2.BORDER_REFLECT)
    local_var = np.maximum(local_sqmean - local_mean * local_mean, 0)
    noise_power = float(local_var.mean())
    gain = local_var / (local_var + noise_power + 1e-8)
    denoised = local_mean + gain * (g - local_mean)
    noise = g - denoised
    flat = noise.ravel()

    H, W = noise.shape
    bs = 16
    Hc, Wc = (H // bs) * bs, (W // bs) * bs
    if Hc >= bs and Wc >= bs:
        blocks = noise[:Hc, :Wc].reshape(Hc // bs, bs, Wc // bs, bs)
        local_vars = blocks.var(axis=(1, 3)).ravel()
    else:
        local_vars = np.array([0.0])

    hp = cv2.Laplacian((g * 255).astype(np.uint8), cv2.CV_64F)
    hp_hist, _ = np.histogram(hp.ravel(), bins=20, density=True)
    hp_hist = hp_hist + 1e-12

    odd_var = float(np.var(noise[::2, :]))
    even_var = float(np.var(noise[1::2, :]))
    col_odd = float(np.var(noise[:, ::2]))
    col_even = float(np.var(noise[:, 1::2]))

    return np.array([
        float(noise.std()), float(kurtosis(flat)), float(skew(flat)),
        float(np.mean(np.abs(flat))), float(np.percentile(np.abs(flat), 90)),
        odd_var / (even_var + 1e-8), col_odd / (col_even + 1e-8),
        float(local_vars.mean()), float(local_vars.std()),
        float(hp.std()), _fast_entropy(hp_hist),
    ], dtype=np.float32)


def _chromatic_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    edge_mask = mag > np.percentile(mag, 70)

    r = img[:, :, 2].astype(np.float32)
    g = img[:, :, 1].astype(np.float32)
    b = img[:, :, 0].astype(np.float32)

    feats = []
    for c1, c2 in [(r, g), (g, b), (r, b)]:
        d = np.abs(c1 - c2)[edge_mask]
        if len(d) == 0:
            feats.extend([0.0, 0.0])
        else:
            feats.extend([float(d.mean()), float(d.std())])

    H, W = gray.shape
    cs = int(min(H, W) * 0.2)
    corners = [(slice(0, cs), slice(0, cs)), (slice(0, cs), slice(W-cs, W)),
               (slice(H-cs, H), slice(0, cs)), (slice(H-cs, H), slice(W-cs, W))]
    corner_diffs = []
    for sy, sx in corners:
        d = np.abs(r[sy, sx] - b[sy, sx])
        corner_diffs.append(d.mean())
    feats.append(float(np.mean(corner_diffs)))
    feats.append(float(np.std(corner_diffs)))

    return np.array(feats, dtype=np.float32)


def _sharpness_features(gray):
    g = gray.astype(np.uint8)
    lap = cv2.Laplacian(g, cv2.CV_64F)
    lap_var = float(lap.var())

    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=3)
    tenengrad = float(np.mean(gx**2 + gy**2))
    sobel_energy = float(np.sum(np.sqrt(gx**2 + gy**2)))

    edges = cv2.Canny(g, 50, 150)
    edge_density = float(edges.mean() / 255.0)

    mag = np.sqrt(gx**2 + gy**2)
    strong = mag > np.percentile(mag, 90)
    edge_width = float(1.0 / (strong.mean() + 1e-8))

    return np.array([lap_var, tenengrad, sobel_energy, edge_density, edge_width], dtype=np.float32)


def _glare_features(img_small):
    hsv = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)

    highlight_mask = v > 240
    highlight_ratio = float(highlight_mask.mean())

    hist, _ = np.histogram(v, bins=32, range=(0, 255), density=True)
    hist = hist + 1e-12
    highlight_entropy = _fast_entropy(hist)

    clip_ratio = float((v >= 254).mean())

    mask_u8 = (highlight_mask.astype(np.uint8)) * 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    glare_areas = stats[1:, cv2.CC_STAT_AREA] if n_labels > 1 else np.array([0])
    glare_mask_area = float(glare_areas.max() / (img_small.shape[0] * img_small.shape[1])) if len(glare_areas) else 0.0

    sat_low = float((s < 30).mean())

    return np.array([highlight_ratio, highlight_entropy, clip_ratio, glare_mask_area, sat_low], dtype=np.float32)


def _colorspace_features(img_small):
    feats = []
    hsv = cv2.cvtColor(img_small, cv2.COLOR_BGR2HSV).astype(np.float32)
    lab = cv2.cvtColor(img_small, cv2.COLOR_BGR2LAB).astype(np.float32)
    ycc = cv2.cvtColor(img_small, cv2.COLOR_BGR2YCrCb).astype(np.float32)
    bgr = img_small.astype(np.float32)

    for space in [bgr, hsv, lab, ycc]:
        for c in range(3):
            ch = space[:, :, c].ravel()
            feats.extend([float(ch.mean()), float(ch.std()), float(skew(ch))])

    b, g, r = cv2.split(bgr)
    rg = r - g
    yb = 0.5 * (r + g) - b
    colorfulness = float(np.sqrt(rg.std()**2 + yb.std()**2) + 0.3 * np.sqrt(rg.mean()**2 + yb.mean()**2))

    corr_rg = float(np.corrcoef(r.ravel(), g.ravel())[0, 1])
    corr_gb = float(np.corrcoef(g.ravel(), b.ravel())[0, 1])

    feats.extend([colorfulness, corr_rg, corr_gb])
    return np.array(feats, dtype=np.float32)


def _extract_features(img_full):
    img_tex = _resize_long_side(img_full, TEXTURE_LONG_SIDE)
    img_stats = _resize_long_side(img_full, STATS_LONG_SIDE)
    gray_tex = cv2.cvtColor(img_tex, cv2.COLOR_BGR2GRAY).astype(np.float64)

    f1 = _wavelet_features(gray_tex)
    f2 = _moire_features(gray_tex)
    f3 = _lbp_features(gray_tex)
    f4 = _noise_features(gray_tex)
    f5 = _chromatic_features(img_tex)
    f6 = _sharpness_features(gray_tex)
    f7 = _glare_features(img_stats)
    f8 = _colorspace_features(img_stats)
    return np.concatenate([f1, f2, f3, f4, f5, f6, f7, f8]).astype(np.float32)


def predict(image_path: str) -> float:
    img = _load_image(image_path)
    feats = _extract_features(img)
    model = _load_model()
    score = model.predict_proba([feats])[0][1]
    return float(score)

if __name__ == "__main__":

    IMAGE_PATH = "/content/WhatsApp Image 2026-06-30 at 5.56.15 PM.jpeg"

    # Warm-up (don't include in timing)
    _ = predict(IMAGE_PATH)

    times = []

    # Measure latency over 20 runs
    for _ in range(20):
        start = time.perf_counter()

        score = predict(IMAGE_PATH)

        end = time.perf_counter()

        times.append((end - start) * 1000)

    print("=" * 50)
    print(f"Prediction Score: {score:.4f}")

    if score >= 0.5:
        print("Prediction: SCREEN / RECAPTURE")
    else:
        print("Prediction: REAL PHOTO")

    print("\nLatency Statistics")
    print(f"Average : {np.mean(times):.2f} ms")
    print(f"Median  : {np.median(times):.2f} ms")
    print(f"Minimum : {np.min(times):.2f} ms")
    print(f"Maximum : {np.max(times):.2f} ms")

    print("=" * 50)


Prediction Score: 0.9192
Prediction: SCREEN / RECAPTURE

Latency Statistics
Average : 180.64 ms
Median  : 178.69 ms
Minimum : 121.65 ms
Maximum : 342.31 ms
